# EV Grid Oracle — GRPO Training (Colab T4)

This notebook trains a small LLM (Qwen 2.5 3B Instruct) with **verifier-based GRPO** on the **real Bangalore road graph** (connected-edge actions only).

- **Environment**: OpenEnv-compatible `EVGridRoadEnvironment` mounted at `/road/` in the Space.
- **Key constraint**: the policy can only choose a **connected neighbor** in the OSM-derived graph (no teleporting).

## Run order (Colab — once per runtime)

1. **Runtime → Change runtime type → GPU** (T4 matches the defaults below).
2. Run the **next code cell** first. It clones this repo, moves into it, and runs `pip install -e .` so `import ev_grid_oracle` works.
3. Run cells **top to bottom**.
4. After training, use the save cell and upload `ev_oracle_lora/` to the Hub (or copy to Drive).

**Links:** [Open in Colab](https://colab.research.google.com/github/NITISH-R-G/ev-grid-oracle/blob/main/training/train_grpo.ipynb) · [Notebook on GitHub](https://github.com/NITISH-R-G/ev-grid-oracle/blob/main/training/train_grpo.ipynb)

**Action schema (strict):**

```text
CURRENT_NODE: <int>
NEXT_NODE: <int>
REASON: max 20 words
CONFIDENCE: 0.0-1.0
```

> If you’re using LoRA/QLoRA, don’t naively upcast a 4-bit base to 16-bit and “merge” at the end without the correct path — it can badly degrade quality. Save adapters cleanly and test post-training inference immediately.


In [ ]:
# Colab setup: clone repo + install package (required for `ev_grid_oracle` imports)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "EV_GRID_ORACLE_REPO",
    "https://github.com/NITISH-R-G/ev-grid-oracle.git",
)
here = Path.cwd().resolve()
if (here / "ev_grid_oracle").is_dir():
    # Already inside a checkout (e.g. local Jupyter opened at repo root)
    root = here
else:
    root = (here / "ev-grid-oracle").resolve()
    if not (root / "ev_grid_oracle").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    os.chdir(root)

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pydantic", "networkx", "openenv-core"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "unsloth", "trl", "transformers", "accelerate", "datasets"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

print("OK — cwd:", Path.cwd().resolve())


In [ ]:
import re
from typing import Optional

from datasets import Dataset

from ev_grid_oracle.road_env import RoadCore
from ev_grid_oracle.road_models import RoadAction, RoadState


core = RoadCore(g=None, nodes=[])  # graph is loaded inside reset()


In [ ]:
ACTION_RE = re.compile(
    r"CURRENT_NODE:\s*(?P<cur>\d+)\s*\n"
    r"NEXT_NODE:\s*(?P<nxt>\d+)\s*\n",
    re.IGNORECASE,
)


def parse_action(text: str) -> Optional[RoadAction]:
    m = ACTION_RE.search(text.strip())
    if not m:
        return None
    try:
        return RoadAction(current_node=int(m.group("cur")), next_node=int(m.group("nxt")))
    except Exception:
        return None


In [ ]:
def _format_neighbors(st: RoadState, *, max_k: int = 12) -> str:
    # Expose valid actions (neighbors) so the LLM can't claim it didn't know.
    g = core.g
    neigh = list(g.neighbors(int(st.node)))[:max_k]
    return ", ".join(str(int(x)) for x in neigh)


def generate_episode_dataset(n: int = 800, *, seed: int = 123) -> Dataset:
    rows = []
    for i in range(n):
        obs = core.reset(seed=seed + i)
        st = obs.state
        neigh = _format_neighbors(st)
        prompt = (
            "You are routing an EV on Bangalore's real road graph. You must pick NEXT_NODE as a connected neighbor only.\n\n"
            f"CURRENT_NODE: {st.node}\n"
            f"BATTERY_PCT: {st.battery_pct_0_100:.1f}\n"
            f"TARGET_STATION_ID: {st.target_station_id}\n"
            f"TARGET_LATLNG: {st.target_lat:.6f},{st.target_lng:.6f}\n"
            f"STEPS_REMAINING: {st.steps_remaining}\n"
            f"VALID_NEXT_NODES: {neigh}\n\n"
            "Respond in this exact schema:\n"
            "CURRENT_NODE: <int>\n"
            "NEXT_NODE: <int>\n"
            "REASON: max 20 words\n"
            "CONFIDENCE: 0.0-1.0\n"
        )
        rows.append({"prompt": prompt, "state_json": st.model_dump(mode="json")})
    return Dataset.from_list(rows)


train_ds = generate_episode_dataset(n=800)
train_ds[0]["prompt"][:450]


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)


In [ ]:
from ev_grid_oracle.road_models import RoadState


def reward_fn(prompts, completions, **kwargs):
    rewards = []

    batch = kwargs.get("batch")
    state_jsons = batch["state_json"] if batch is not None and "state_json" in batch else None

    for completion, state_json in zip(completions, state_jsons or [None] * len(completions)):
        if state_json is None:
            rewards.append(0.0)
            continue

        st = RoadState.model_validate(state_json)
        action = parse_action(completion)
        if action is None:
            rewards.append(-1.0)
            continue

        # Hard anti-cheat: must match the provided current node.
        if int(action.current_node) != int(st.node):
            rewards.append(-1.0)
            continue

        # Step local env from the same state.
        local = RoadCore(g=core.g, nodes=core.nodes)
        local.node = int(st.node)
        local.battery_pct = float(st.battery_pct_0_100)
        local.target_station_id = str(st.target_station_id)
        local.steps_remaining = int(st.steps_remaining)

        obs = local.step(action)
        base_r = float(obs.reward_breakdown.get("total", 0.0))

        # Penalize any anti-cheat flags from the verifier.
        cheat_pen = -1.0 if obs.anti_cheat_flags else 0.0

        rewards.append(base_r + cheat_pen)

    return rewards


In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="ev_oracle_grpo_road",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_generations=4,
    max_completion_length=120,
    report_to=["tensorboard"],  # logs under output_dir; see next markdown cell + docs/submission/training-artifacts-and-logs.md
    logging_steps=1,
)

# Minimal guardrail sampling: print a few raw generations early.
class SampleCallback:
    def __init__(self, every_steps: int = 10, n: int = 3):
        self.every_steps = every_steps
        self.n = n

    def on_step_end(self, args, state, control, **kwargs):
        step = int(getattr(state, "global_step", 0) or 0)
        if step == 1 or (self.every_steps and step % self.every_steps == 0):
            ex = train_ds.select(range(min(self.n, len(train_ds))))
            for i, p in enumerate(ex["prompt"]):
                out = tokenizer.decode(
                    model.generate(
                        **tokenizer(p, return_tensors="pt").to(model.device),
                        max_new_tokens=80,
                        do_sample=True,
                        temperature=0.7,
                    )[0],
                    skip_special_tokens=True,
                )
                print(f"\n--- sample step={step} i={i} ---\n", out[-400:])
        return control


trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=train_ds,
    callbacks=[SampleCallback(every_steps=25, n=2)],
)

trainer.train()


## After training — logs for judges

Scalars go to **`ev_oracle_grpo_road/`** (same as `GRPOConfig.output_dir`): TensorBoard event files + `trainer_state.json`.

**In Colab**, open TensorBoard (run in a new cell *after* `trainer.train()` finishes):

```python
%load_ext tensorboard
%tensorboard --logdir ev_oracle_grpo_road
```

**For submission:** run `python tools/export_grpo_tensorboard_plots.py --logdir ev_oracle_grpo_road --out-dir artifacts` after copying the run folder from Colab (needs `tensorboard` + `matplotlib`), **or** screenshot **reward** and **loss** → `artifacts/grpo_reward.png` / `artifacts/grpo_loss.png`, or paste a short console tail into `artifacts/training_logs/colab_console_tail.txt`. Full checklist: [`docs/submission/training-artifacts-and-logs.md`](https://github.com/NITISH-R-G/ev-grid-oracle/blob/main/docs/submission/training-artifacts-and-logs.md).

In [ ]:
# Save adapters (recommended) and a merged artifact (optional)
model.save_pretrained("ev_oracle_lora")
tokenizer.save_pretrained("ev_oracle_lora")

# Optional merged export (test immediately after):
# model.save_pretrained_merged("ev_oracle_merged_16bit", tokenizer, save_method="merged_16bit")
